# 07 — Preprocessor LOO Ablation

Quantify each preprocessor component's contribution by removing it while keeping the rest.

**Configs (per noise type):**

| Noise  | Components                                            | Configs |
|--------|--------------------------------------------------------|---------|
| OCR    | charfix, spellcorrect                                  | all, -charfix, -spellcorrect |
| ASR    | truecase, homophone                                    | all, -truecase, -homophone |
| Social | unrepeat, unabbrev, spellcorrect                       | all, -unrepeat, -unabbrev, -spellcorrect |

= 3 + 3 + 4 = **10 configs** per model × 4 models = **40 eval passes**.

Each model's ablation was run in an isolated subprocess (see
`scripts/eval_ablation.py`) writing one JSON per model to
`results/tables/ablation_partials/<safe_model>.json`.

```
for m in bert-base-uncased bert-base-cased gpt2 google/byt5-small; do
    python scripts/eval_ablation.py "$m"
done
```

**Interpretation:** `contribution_of(component) = F1(all) - F1(-component)`. Large
positive values mean the component matters; near-zero means the component is
redundant or harmful; negative means removing it actually *helped* (overcorrection).

In [1]:
import json
import os

import pandas as pd

ROOT = os.path.dirname(os.getcwd())
TABLES_DIR = os.path.join(ROOT, "results", "tables")
PARTIALS_DIR = os.path.join(TABLES_DIR, "ablation_partials")

MODEL_NAMES = [
    "bert-base-uncased",
    "bert-base-cased",
    "gpt2",
    "google/byt5-small",
]

In [2]:
records = []
for name in MODEL_NAMES:
    safe = name.replace("/", "__")
    with open(os.path.join(PARTIALS_DIR, f"{safe}.json"), "r", encoding="utf-8") as f:
        records.extend(json.load(f))

ab_df = pd.DataFrame(records)[["model", "noise", "config", "f1", "precision", "recall"]]
ab_df.to_csv(os.path.join(TABLES_DIR, "preprocess_ablation.csv"), index=False)
print(f"Saved {os.path.join(TABLES_DIR, 'preprocess_ablation.csv')}")
ab_df

Saved /Users/narly/Code/Study/S26/NLP/S26-NLP-Tokenization-for-Noisy-Texts/results/tables/preprocess_ablation.csv


,model,noise,config,f1,precision,recall
0,bert-base-uncased,ocr,all,0.7829,0.7585,0.8089
1,bert-base-uncased,ocr,-charfix,0.6654,0.7232,0.6161
2,bert-base-uncased,ocr,-spellcorrect,0.7537,0.7012,0.8146
3,bert-base-uncased,asr,all,0.8297,0.8274,0.8320
4,bert-base-uncased,asr,-truecase,0.8297,0.8274,0.8320
5,bert-base-uncased,asr,-homophone,0.8255,0.8230,0.8281
6,bert-base-uncased,social,all,0.7318,0.6763,0.7973
7,bert-base-uncased,social,-unrepeat,0.7276,0.6713,0.7943
8,bert-base-uncased,social,-unabbrev,0.7320,0.6766,0.7973
9,bert-base-uncased,social,-spellcorrect,0.6075,0.5173,0.7357


## 1. Per-component contribution

`contribution = F1(all) - F1(-component)`

In [3]:
# F1 baseline (all components) per (model, noise)
all_df = ab_df[ab_df["config"] == "all"][["model", "noise", "f1"]].rename(columns={"f1": "f1_all"})

# F1 per (model, noise, -component)
minus_df = ab_df[ab_df["config"] != "all"].copy()
minus_df["component"] = minus_df["config"].str.lstrip("-")
minus_df.rename(columns={"f1": "f1_minus"}, inplace=True)

contrib = minus_df.merge(all_df, on=["model", "noise"])
contrib["contribution"] = (contrib["f1_all"] - contrib["f1_minus"]).round(4)
contrib = contrib[["model", "noise", "component", "f1_all", "f1_minus", "contribution"]]
contrib = contrib.sort_values(["model", "noise", "component"]).reset_index(drop=True)
contrib.to_csv(os.path.join(TABLES_DIR, "preprocess_contributions.csv"), index=False)
print(f"Saved preprocess_contributions.csv")
contrib

Saved preprocess_contributions.csv


,model,noise,component,f1_all,f1_minus,contribution
0,bert-base-cased,asr,homophone,0.6746,0.6757,-0.0011
1,bert-base-cased,asr,truecase,0.6746,0.2018,0.4728
2,bert-base-cased,ocr,charfix,0.7993,0.7055,0.0938
3,bert-base-cased,ocr,spellcorrect,0.7993,0.8086,-0.0093
4,bert-base-cased,social,spellcorrect,0.7394,0.7105,0.0289
5,bert-base-cased,social,unabbrev,0.7394,0.7395,-0.0001
6,bert-base-cased,social,unrepeat,0.7394,0.7369,0.0025
7,bert-base-uncased,asr,homophone,0.8297,0.8255,0.0042
8,bert-base-uncased,asr,truecase,0.8297,0.8297,0.0000
9,bert-base-uncased,ocr,charfix,0.7829,0.6654,0.1175


## 2. Heatmap-ready pivots

In [4]:
for noise in ["ocr", "asr", "social"]:
    print(f"\n=== {noise.upper()}: F1(all) - F1(-component) per model × component ===")
    sub = contrib[contrib["noise"] == noise]
    pivot = sub.pivot(index="model", columns="component", values="contribution")
    print(pivot.to_string())


=== OCR: F1(all) - F1(-component) per model × component ===
component          charfix  spellcorrect
model                                   
bert-base-cased     0.0938       -0.0093
bert-base-uncased   0.1175        0.0292
google/byt5-small   0.0887       -0.0165
gpt2                0.0519       -0.0048

=== ASR: F1(all) - F1(-component) per model × component ===
component          homophone  truecase
model                                 
bert-base-cased      -0.0011    0.4728
bert-base-uncased     0.0042    0.0000
google/byt5-small    -0.0087    0.2263
gpt2                  0.0000    0.4298

=== SOCIAL: F1(all) - F1(-component) per model × component ===
component          spellcorrect  unabbrev  unrepeat
model                                              
bert-base-cased          0.0289   -0.0001    0.0025
bert-base-uncased        0.1243   -0.0002    0.0042
google/byt5-small        0.0276    0.0000    0.0033
gpt2                     0.0135   -0.0002    0.0002


## 3. Ranking: which component matters most?

In [5]:
# Dominant (largest positive contribution) per (model, noise)
rank = contrib.loc[
    contrib.groupby(["model", "noise"])["contribution"].idxmax()
][["model", "noise", "component", "contribution"]].reset_index(drop=True)
rank.columns = ["model", "noise", "dominant_component", "contribution"]
print("Dominant preprocessor component per (model, noise):")
rank

Dominant preprocessor component per (model, noise):


,model,noise,dominant_component,contribution
0,bert-base-cased,asr,truecase,0.4728
1,bert-base-cased,ocr,charfix,0.0938
2,bert-base-cased,social,spellcorrect,0.0289
3,bert-base-uncased,asr,homophone,0.0042
4,bert-base-uncased,ocr,charfix,0.1175
5,bert-base-uncased,social,spellcorrect,0.1243
6,google/byt5-small,asr,truecase,0.2263
7,google/byt5-small,ocr,charfix,0.0887
8,google/byt5-small,social,spellcorrect,0.0276
9,gpt2,asr,truecase,0.4298


## 4. Negative contributions (overcorrection)

Rows where removing the component INCREASED F1 — the component is net-harmful in context.

In [6]:
harm = contrib[contrib["contribution"] < 0].copy()
harm = harm.sort_values("contribution").reset_index(drop=True)
print("Components that hurt F1 when included (negative contribution = helpful to remove):")
harm

Components that hurt F1 when included (negative contribution = helpful to remove):


,model,noise,component,f1_all,f1_minus,contribution
0,google/byt5-small,ocr,spellcorrect,0.7843,0.8008,-0.0165
1,bert-base-cased,ocr,spellcorrect,0.7993,0.8086,-0.0093
2,google/byt5-small,asr,homophone,0.5901,0.5988,-0.0087
3,gpt2,ocr,spellcorrect,0.5970,0.6018,-0.0048
4,bert-base-cased,asr,homophone,0.6746,0.6757,-0.0011
5,bert-base-uncased,social,unabbrev,0.7318,0.7320,-0.0002
6,gpt2,social,unabbrev,0.5431,0.5433,-0.0002
7,bert-base-cased,social,unabbrev,0.7394,0.7395,-0.0001
